In [1]:
import pandas as pd
import os
from zipfile import ZipFile
import io
from urllib.request import urlopen
import geopandas as gpd

#--------------------------------------------
# configs
#--------------------------------------------
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)

pums_years = [2012,2013,2014,2015,2016,2017,2018,2019, 2021, 2022, 2023, 2024]
state_abbr = "wa"

In [2]:
# download pums files if not already present
for pums_table in ["p", "h"]:
    for year in pums_years:
        out_name = f"psam_{pums_table}{year}.csv"
        out_path = os.path.join(data_dir, out_name)
        if os.path.exists(out_path):
            continue

        pums_url = (
            f"https://www2.census.gov/programs-surveys/acs/data/pums/"
            f"{year}/1-Year/csv_{pums_table}{state_abbr}.zip"
        )
        r = urlopen(pums_url).read()
        with ZipFile(io.BytesIO(r)) as archive:
            inner = next(n for n in archive.namelist() if n.lower().endswith(".csv"))
            with archive.open(inner) as src, open(out_path, "wb") as dst:
                dst.write(src.read())

In [ ]:
# create puma to county crosswalks
def create_puma_county_lookup(puma_year,tiger_year):
    # create county geoid list
    state = 53
    counties = ['035', '033', '053', '061']
    county_geoids_str = [f'{state}{str(c).zfill(3)}' for c in counties]
    puma_year_2digit = str(puma_year)[-2:]

    # download county shapefile from tigerweb
    cnty = (
        gpd.read_file(f'https://www2.census.gov/geo/tiger/TIGER{tiger_year}/COUNTY/tl_{tiger_year}_us_county.zip')
        .query('GEOID.isin(@county_geoids_str)')
        .assign(GEOID =lambda df: df['GEOID'].astype(int))
        .rename(columns={'GEOID':'county_id'})
        [['county_id', 'geometry']]
    )

    # download puma shapefile from tigerweb
    puma_folder = f"PUMA{puma_year_2digit}" if int(tiger_year) >= 2022 else "PUMA"
    puma = (
        gpd.read_file(
            f"https://www2.census.gov/geo/tiger/TIGER{tiger_year}/{puma_folder}/"
            f"tl_{tiger_year}_53_puma{puma_year_2digit}.zip"
        )
        .assign(puma_id=lambda df: df[f"GEOID{puma_year_2digit}"].astype(int))
        [[f"PUMACE{puma_year_2digit}", "geometry"]]
    )

    # get centroids of pumas
    puma.geometry = puma.representative_point()

    # spatial join pumas to counties
    puma_cnty = gpd.sjoin(puma, cnty, how='inner', predicate='within').drop(columns='index_right')

    # create region column
    puma_cnty['region'] = 1

    # save to csv in data directory
    puma_cnty[[f'PUMACE{puma_year_2digit}', 'county_id','region']].to_csv(os.path.join(data_dir, f'puma{puma_year_2digit}_cnty_xwalk.csv'), index=False)

create_puma_county_lookup(puma_year=2010, tiger_year=2020)
create_puma_county_lookup(puma_year=2020, tiger_year=2025)

In [3]:
# load puma to county crosswalks
xwalk10 = (
    pd.read_csv(os.path.join(data_dir, 'puma10_cnty_xwalk.csv'))
    .rename(columns={'PUMACE10':'PUMA'})
    .set_index('PUMA')['county_id']
)
xwalk20 = (
    pd.read_csv(os.path.join(data_dir, 'puma20_cnty_xwalk.csv'))
    .rename(columns={'PUMACE20':'PUMA'})
    .set_index('PUMA')['county_id']
)

In [4]:
county_ids = [35061, 53033, 53035, 53053]

df = None
for pums_year in pums_years:
    if pums_year >=2012 and pums_year <= 2021:
        xwalk = xwalk10
    elif pums_year >= 2022 and pums_year <= 2031:
        xwalk = xwalk20


    if pums_year > 2020:
        hh_type = 'TYPEHUGQ'
    else:
        hh_type = 'TYPE'

    p = (
        pd.read_csv(os.path.join(data_dir, f'psam_p{pums_year}.csv'), usecols=['PUMA','SERIALNO','SPORDER','AGEP','PWGTP'])
        .assign(county_id = lambda df: df['PUMA'].map(xwalk))
        .query('county_id.isin(@county_ids) & PWGTP > 0')
    )

    h = (
        pd.read_csv(os.path.join(data_dir, f'psam_h{pums_year}.csv'), usecols=['PUMA',hh_type,'SERIALNO','WGTP'])
        .query('WGTP > 0')
    )
    if pums_year > 2020:
        h = h[h['TYPEHUGQ'].isin([1])]
    else:
        h = h[h['TYPE'].isin([1,2])]

    # Filter for households without group quarters
    p = p[p['SERIALNO'].isin(h['SERIALNO'])]

    # create 10 year age groups using starting with ages_0_14 and ending with ages_85_plus
    p['age_group'] = pd.cut(
        p['AGEP'],
        bins=[0, 14, 24, 34, 44, 54, 64, 74, 84, float('inf')],
        labels=['ages_0_14', 'ages_15_24', 'ages_25_34', 'ages_35_44', 'ages_45_54', 'ages_55_64', 'ages_65_74', 'ages_75_84', 'ages_85_plus']
    )

    # calculate headship rates by age group
    head = p.loc[p['SPORDER'] == 1].groupby(['age_group'])['PWGTP'].sum()
    pop = p.groupby(['age_group'])['PWGTP'].sum()
    headship_rates = (head / pop).fillna(0)
    headship_rates = headship_rates.to_frame().reset_index().rename(columns={'PWGTP':pums_year})
    if df is None:
        df = headship_rates
    else:
        df = df.merge(headship_rates, on=['age_group'], how='outer')


In [5]:
df.to_parquet(os.path.join(data_dir, 'pums_1yr_headship_rates.parquet'), index=False)